# Milestone 1: Data Exploration and Preprocessing

The purpose of this notebook is to:

- load and inspect small samples of the Amazon Reviews 2023 dataset
- examine both **review** and **metadata** records
- identify the most useful fields for retrieval
- justify preprocessing choices
- create a compact cleaned dataset for downstream **BM25** and **semantic retrieval**

For this milestone, the analysis begins with two sampled categories:

- **All_Beauty**
- **Health_and_Personal_Care**

After comparing them, one category is selected for the first retrieval system.

This notebook is intentionally focused and lightweight with 200 entries in each category.


## Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd

SRC_DIR = Path.cwd().resolve().parent / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
    
from preprocessing import (
    find_repo_root,
    load_jsonl,
    summarize_columns,
    simple_clean,
    validate_retrieval_dataframe
)

In [2]:
repo_root = find_repo_root()
repo_root

WindowsPath('c:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray')

## Locate sample files

This notebook assumes small sampled JSONL files are stored under `data/processed/` which is the case with out project structure.

For EDA, both **review** and **metadata** samples should be inspected. This is important because retrieval design depends on understanding:
- what the review text contains
- what useful contextual product metadata is available
- how those two sources can be combined into a better retrieval document

In [3]:
sample_dir = repo_root / "data" / "processed"

paths = {
    "all_beauty_reviews": sample_dir / "sample_All_Beauty.jsonl",
    "all_beauty_meta": sample_dir / "sample_meta_All_Beauty.jsonl",
    "health_reviews": sample_dir / "sample_Health_and_Personal_Care.jsonl",
    "health_meta": sample_dir / "sample_meta_Health_and_Personal_Care.jsonl",
}

for name, path in paths.items():
    print(f"{name}: {path} | exists={path.exists()}")

all_beauty_reviews: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_All_Beauty.jsonl | exists=True
all_beauty_meta: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_meta_All_Beauty.jsonl | exists=True
health_reviews: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_Health_and_Personal_Care.jsonl | exists=True
health_meta: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_meta_Health_and_Personal_Care.jsonl | exists=True


In [4]:
# Load sampled review and metadata files for both candidate categories, then report each dataset's shape.
all_beauty_reviews = load_jsonl(paths["all_beauty_reviews"])
all_beauty_meta = load_jsonl(paths["all_beauty_meta"])
health_reviews = load_jsonl(paths["health_reviews"])
health_meta = load_jsonl(paths["health_meta"])

datasets = {
    "All_Beauty reviews": all_beauty_reviews,
    "All_Beauty metadata": all_beauty_meta,
    "Health_and_Personal_Care reviews": health_reviews,
    "Health_and_Personal_Care metadata": health_meta,
}

for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")

All_Beauty reviews: shape=(200, 10)
All_Beauty metadata: shape=(200, 14)
Health_and_Personal_Care reviews: shape=(200, 10)
Health_and_Personal_Care metadata: shape=(200, 14)


In [5]:
# Display the first record from each non-empty dataset to inspect its structure and example field values.
for name, df in datasets.items():
    if df.empty:
        continue
    print(f"\n{name} sample record:")
    print(df.iloc[0].to_dict())


All_Beauty reviews sample record:
{'rating': 5, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': Timestamp('2020-05-05 14:08:48.923000'), 'helpful_vote': 0, 'verified_purchase': True}

All_Beauty metadata sample record:
{'main_category': 'All Beauty', 'title': 'Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)', 'average_rating': 4.8, 'rating_number': 10, 'features': [], 'description': [], 'price': nan, 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg', 'vari

## Dataset overview

The Amazon Reviews 2023 dataset contains separate files for:
- **reviews**, which contain user-written review text and ratings
- **metadata**, which contain product-level attributes such as title, categories, descriptions, and features

For this milestone, these two sources are examined together because retrieval quality may improve when review text is enriched with product metadata.

In [6]:
# Summarize each dataset's size and available columns.
overview_rows = []

for name, df in datasets.items():
    overview_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": ", ".join(df.columns.tolist())
    })

overview_df = pd.DataFrame(overview_rows)
overview_df

,dataset,rows,columns,column_names
0,All_Beauty reviews,200,10,"rating, title, text, images, asin, parent_asin..."
1,All_Beauty metadata,200,14,"main_category, title, average_rating, rating_n..."
2,Health_and_Personal_Care reviews,200,10,"rating, title, text, images, asin, parent_asin..."
3,Health_and_Personal_Care metadata,200,14,"main_category, title, average_rating, rating_n..."


## Inspect sample records

Sample records are printed below to understand the structure of both review files and metadata files.

This step helps determine:
- which columns are consistently populated
- which fields are likely useful for retrieval
- whether metadata fields should be merged with reviews during document construction

In [7]:
for name, df in datasets.items():
    if not df.empty:
        print(f"\n{name} — first record")
        display(df.head(1))


All_Beauty reviews — first record


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True



All_Beauty metadata — first record


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,NaN



Health_and_Personal_Care reviews — first record


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,12 mg is 12 on the periodic table people! Mg f...,This review is more to clarify someone else’s ...,[],B07TDSJZMR,B07TDSJZMR,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-02-06 00:49:35.902,3,True



Health_and_Personal_Care metadata — first record


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Health & Personal Care,Silicone Bath Body Brush Exfoliator Shower Bac...,3.9,7,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Rzoeox,[],{'Package Dimensions': '15 x 3.3 x 1.5 inches;...,B07V346GZH,NaN


## Quick field inspection

Next, we inspect important columns and missingness patterns. This helps identify fields that are useful and practical for retrieval.

In [8]:
for name, df in datasets.items():
    if not df.empty:
        print(f"\n{name}")
        display(summarize_columns(df))


All_Beauty reviews


,column,non_null_count,null_count,dtype,non_null_pct
0,rating,200,0,int64,1.0
1,title,200,0,str,1.0
2,text,200,0,str,1.0
3,images,200,0,object,1.0
4,asin,200,0,str,1.0
5,parent_asin,200,0,str,1.0
6,user_id,200,0,str,1.0
7,timestamp,200,0,datetime64[ms],1.0
8,helpful_vote,200,0,int64,1.0
9,verified_purchase,200,0,bool,1.0



All_Beauty metadata


,column,non_null_count,null_count,dtype,non_null_pct
0,main_category,200,0,str,1.000
1,title,200,0,str,1.000
2,average_rating,200,0,float64,1.000
3,rating_number,200,0,int64,1.000
4,features,200,0,object,1.000
5,description,200,0,object,1.000
6,images,200,0,object,1.000
7,videos,200,0,object,1.000
8,details,200,0,object,1.000
9,categories,200,0,object,1.000



Health_and_Personal_Care reviews


,column,non_null_count,null_count,dtype,non_null_pct
0,rating,200,0,int64,1.0
1,title,200,0,str,1.0
2,text,200,0,str,1.0
3,images,200,0,object,1.0
4,asin,200,0,str,1.0
5,parent_asin,200,0,str,1.0
6,user_id,200,0,str,1.0
7,timestamp,200,0,datetime64[ms],1.0
8,helpful_vote,200,0,int64,1.0
9,verified_purchase,200,0,bool,1.0



Health_and_Personal_Care metadata


,column,non_null_count,null_count,dtype,non_null_pct
0,main_category,200,0,str,1.00
1,title,200,0,str,1.00
2,average_rating,200,0,float64,1.00
3,rating_number,200,0,int64,1.00
4,features,200,0,object,1.00
5,description,200,0,object,1.00
6,images,200,0,object,1.00
7,videos,200,0,object,1.00
8,details,200,0,object,1.00
9,categories,200,0,object,1.00


### Observations from review and metadata samples

The review samples for both **All_Beauty** and **Health_and_Personal_Care** are structurally consistent. In both categories, the review files contain fully populated fields for `rating`, `title`, `text`, `asin`, `parent_asin`, `timestamp`, `helpful_vote`, and `verified_purchase`. This suggests that the review files provide a reliable core source of retrieval text, with `text` as the main document body and `rating` and `title` as useful supporting fields for display and interpretation.

The metadata samples are also largely complete across both categories. Fields such as `main_category`, `title`, `average_rating`, `rating_number`, `features`, `description`, `categories`, and `parent_asin` are fully populated in the sampled rows. These fields are useful because they add product-level context that is not always present in the reviews themselves. In particular, `description`, `features`, and `categories` appear especially valuable for enriching retrieval documents, since they may improve both keyword matching and semantic retrieval.

Some metadata fields appear less useful for Milestone 1. For example, `price` has many missing values in both categories, and `bought_together` is entirely missing in the sampled data. Similarly, fields such as `images`, `videos`, and `details` may contain useful information in some contexts, but they are less directly relevant for a first-pass text retrieval system. As a result, the retrieval pipeline will prioritize textual review content together with the most informative metadata fields, while excluding sparse or less immediately useful attributes.

## Candidate retrieval fields

For retrieval, the most useful fields are those that provide either:
1. searchable text for matching user queries, or
2. metadata that improves result context and display.

In this dataset, the review file provides the main retrieval text, while the metadata file adds product-level context such as title, description, features, and categories.

In [9]:
candidate_columns = [
    "asin",
    "parent_asin",
    "title",
    "text",
    "rating",
    "description",
    "features",
    "categories",
    "average_rating",
]

for name, df in datasets.items():
    if df.empty:
        continue
    
    available = [col for col in candidate_columns if col in df.columns]
    print(f"\n{name} candidate columns:")
    print(available)


All_Beauty reviews candidate columns:
['asin', 'parent_asin', 'title', 'text', 'rating']

All_Beauty metadata candidate columns:
['parent_asin', 'title', 'description', 'features', 'categories', 'average_rating']

Health_and_Personal_Care reviews candidate columns:
['asin', 'parent_asin', 'title', 'text', 'rating']

Health_and_Personal_Care metadata candidate columns:
['parent_asin', 'title', 'description', 'features', 'categories', 'average_rating']


## Category comparison and milestone choice

Two categories were sampled for initial inspection:

- **All_Beauty**
- **Health_and_Personal_Care**

Both categories appear suitable for retrieval experiments. In each case, the review samples provide the main searchable review text through `text`, along with identifiers and review ratings, while the metadata samples provide useful product context through `title`, `description`, `features`, and `categories`. This means that either category could support both keyword-based retrieval and semantic search.

For Milestone 1, however, to keep the retrieval pipeline simpler and the evaluation easier to interpret, **All_Beauty** is selected as the primary category for the remainder of this milestone. It offers a manageable scope and supports clear natural-language product queries, making it a good fit for comparing BM25 and embedding-based retrieval.

In [10]:
reviews_df = all_beauty_reviews.copy()
meta_df = all_beauty_meta.copy()

print("All_Beauty reviews shape:", reviews_df.shape)
print("All_Beauty metadata shape:", meta_df.shape)

All_Beauty reviews shape: (200, 10)
All_Beauty metadata shape: (200, 14)


## Inspect selected category in more detail

In [11]:
display(reviews_df.head(3))
display(meta_df.head(3))

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,NaN
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,NaN
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],{'Manufacturer': 'Levine Health Products'},B000B658RI,NaN


## Join strategy

Here, the review and metadata files are linked using the shared product identifier `parent_asin`.

This join is important because the two datasets provide complementary information. The review file contains the main user-authored retrieval text, while the metadata file contributes product-level context such as the product title and other descriptive attributes when available. Combining them allows each retrieval document to include both opinion-based review content and supporting product context.

This merged representation is expected to improve retrieval quality compared with using review text alone, particularly for short or vague reviews where metadata can make the product intent clearer.

In [12]:
# Use the shared product identifier to link review records with product metadata.
join_col = "parent_asin"
join_col

'parent_asin'

In [13]:
# Merge review data with metadata using the shared product identifier.
merged_df = reviews_df.merge(
    meta_df,
    on=join_col,
    how="left",
    suffixes=("_review", "_meta")
)

print("Merged shape:", merged_df.shape)
display(merged_df.head())

Merged shape: (200, 23)


,rating,title_review,text,images_review,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,rating_number,features,description,price,images_meta,videos,store,categories,details,bought_together
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Observations after merging reviews with metadata

The merged sample shows that many review records do not have matching metadata values, as several metadata fields appear as `NaN` after the join. This suggests that, in the sampled data, metadata enrichment is not available for every review. As a result, the retrieval pipeline should treat the review fields as the primary source of searchable content, while using metadata only as supplementary context when it is present.

This also supports using a left join: all review records are preserved for retrieval, even when corresponding metadata is missing.

## Field selection for retrieval

The retrieval dataset should remain compact while preserving the information most useful for search and result display.

### Selected fields

- **doc_id**: unique identifier for each retrieval document
- **asin / parent_asin**: product identifiers used for linking and grouping
- **title**: short contextual text for retrieval and display
- **review_text**: main searchable review content
- **rating**: review score for result display

### Justification

The review `text` is the primary retrieval signal because it contains the main user-authored content and is consistently available in the sampled review data. The `title` field provides short contextual text that can help with matching brief queries and interpreting results. The identifiers `asin` and `parent_asin` are retained to support linking and grouping, while `rating` is kept for result display. Metadata fields such as `description`, `features`, and `categories` were examined during EDA, but they were excluded from the final retrieval dataset because they were not reliably populated after merging.

In [14]:
# Build a compact retrieval table with review text as the core content
# and metadata as optional enrichment.
retrieval_df = pd.DataFrame({
    "parent_asin": merged_df["parent_asin"],
    "asin": merged_df["asin"],
    "title": merged_df["title_review"],
    "review_text": merged_df["text"],
    "rating": merged_df["rating"],
})

In [15]:

retrieval_df.tail(10)

,parent_asin,asin,title,review_text,rating
190,B07MW1TBSN,B07MW1TBSN,Product to thick,The product was okay a little to thick to try ...,3
191,B07KZQDM8Y,B07KZQDM8Y,I left a negative review earlier,I promised to retract the negative review once...,3
192,B0B8F6MWFJ,B0B8F6MWFJ,This fragrance is neither offensive nor entici...,I wish I loved this. This is a nicely packaged...,3
193,B09ZKWV5MK,B09ZKWV5MK,Typical thin and slinky silk scrunchies,There are six silk scrunchies in this package ...,4
194,B08HVRP54L,B08HVRP54L,The cream is blue but goes on clear; it is ver...,The photo on Amazon must be wrong. This cream ...,3
195,B08WPXVK2P,B08WPXVK2P,"Thick, fragrance-free diaper cream; absorbs in...",The flip top cap has a tab on it that has to b...,4
196,B0B5X48DZR,B09QHM6FKK,Good detangling comb for the right pet; handle...,I love the tines on this long and short detang...,3
197,B08N5NDVGH,B08N5NDVGH,Light-weight cream that is a bit tacky to star...,This comes in a one gallon jug. There are two ...,3
198,B09P144WHW,B09P144WHW,Not totally easy clean brush & tines do not pu...,I used this on my three long-haired cats and i...,2
199,B09QT8SLJB,B09QT8SLJB,Great medium-density blenders; flat angle is p...,This is a nice set of five individually wrappe...,3


In [16]:
retrieval_df.head(10)

,parent_asin,asin,title,review_text,rating
0,B00YQ6X8EO,B00YQ6X8EO,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,5
1,B081TJ8YS3,B081TJ8YS3,Works great but smells a little weird.,"This product does what I need it to do, I just...",4
2,B097R46CSY,B07PNNCSP9,Yes!,"Smells good, feels great!",5
3,B09JS339BZ,B09JS339BZ,Synthetic feeling,Felt synthetic,1
4,B08BZ63GMJ,B08BZ63GMJ,A+,Love it,5
5,B00R8DXL44,B00R8DXL44,Pretty Color,The polish was quiet thick and did not apply s...,4
6,B099DRHW5V,B099DRHW5V,Handy,Great for many tasks. I purchased these for m...,5
7,B08BBQ29N5,B088SZDGXG,Meh,These were lightweight and soft but much too s...,3
8,B08P2DZB4X,B08P2DZB4X,Great for at home use and so easy to use!,This is perfect for my between salon visits. I...,5
9,B086QY6T7N,B086QY6T7N,Nice shampoo for the money,I get Keratin treatments at the salon at least...,5


In [17]:
# Create a unique document identifier for each retrieval record.
retrieval_df["doc_id"] = retrieval_df["parent_asin"].astype(str) + "_" + retrieval_df.index.astype(str)

retrieval_df = retrieval_df[
    ["doc_id", "parent_asin", "asin", "title", "review_text", "rating"]
].copy()

retrieval_df.head()

,doc_id,parent_asin,asin,title,review_text,rating
0,B00YQ6X8EO_0,B00YQ6X8EO,B00YQ6X8EO,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,5
1,B081TJ8YS3_1,B081TJ8YS3,B081TJ8YS3,Works great but smells a little weird.,"This product does what I need it to do, I just...",4
2,B097R46CSY_2,B097R46CSY,B07PNNCSP9,Yes!,"Smells good, feels great!",5
3,B09JS339BZ_3,B09JS339BZ,B09JS339BZ,Synthetic feeling,Felt synthetic,1
4,B08BZ63GMJ_4,B08BZ63GMJ,B08BZ63GMJ,A+,Love it,5


## Text preprocessing decisions

The preprocessing is intentionally minimal so that the text remains suitable for both BM25 and embedding-based retrieval.

### Goals
- preserve useful lexical information for **BM25**
- preserve semantic meaning for **embedding-based retrieval**
- remove obvious noise
- avoid aggressive normalization too early

### Applied preprocessing
- lowercase text
- remove HTML tags
- remove URLs
- normalize whitespace
- combine `title` and `review_text` into a single retrieval field

### Not applied at this stage
- stemming
- lemmatization
- stopword removal
- heavy punctuation stripping

These more aggressive steps are avoided because they may remove useful retrieval signal, particularly for product-specific wording and semantically meaningful phrases.

In [18]:
# Combine title and review text, then apply lightweight cleaning for retrieval.
retrieval_df["combined_text"] = (
    retrieval_df["title"].fillna("") + " " + retrieval_df["review_text"].fillna("")
)

retrieval_df["text_clean"] = retrieval_df["combined_text"].apply(simple_clean)

retrieval_df = retrieval_df.rename(columns={"text_clean": "text"})

retrieval_df[["doc_id", "combined_text", "text"]].head()

,doc_id,combined_text,text
0,B00YQ6X8EO_0,Such a lovely scent but not overpowering. This...,such a lovely scent but not overpowering. this...
1,B081TJ8YS3_1,Works great but smells a little weird. This pr...,works great but smells a little weird. this pr...
2,B097R46CSY_2,"Yes! Smells good, feels great!","yes! smells good, feels great!"
3,B09JS339BZ_3,Synthetic feeling Felt synthetic,synthetic feeling felt synthetic
4,B08BZ63GMJ_4,A+ Love it,a+ love it


In [19]:
validate_retrieval_dataframe(
    retrieval_df[["doc_id", "parent_asin", "asin", "title", "rating", "text"]]
)

The retrieval text is constructed by combining the review `title` and `review_text`. This is useful because the title often acts as a short summary, while the review text provides the main content. The resulting `combined_text` therefore captures slightly more context than either field alone.

The cleaned field, `text_clean`, preserves the main meaning of the original review while normalizing casing and formatting. This makes it a suitable input for both BM25 and semantic retrieval.

## Basic quality checks

A few simple quality checks are reported below to confirm that the cleaned retrieval dataset is usable for downstream indexing and search.

In [20]:
quality_checks = pd.DataFrame(
    {
        "metric": [
            "number_of_documents",
            "missing_title",
            "missing_review_text",
            "empty_text",
        ],
        "value": [
            len(retrieval_df),
            retrieval_df["title"].fillna("").str.strip().eq("").sum(),
            retrieval_df["review_text"].fillna("").str.strip().eq("").sum(),
            retrieval_df["text"].fillna("").str.strip().eq("").sum(),
        ],
    }
)

quality_checks

,metric,value
0,number_of_documents,200
1,missing_title,0
2,missing_review_text,0
3,empty_text,0


## Export cleaned dataset

The cleaned dataset is exported for downstream retrieval scripts.

Two formats are saved:

- **Parquet** for efficient reuse in Python-based retrieval pipelines
- **JSONL** for simple document-oriented loading in later stages of the project

In [21]:
# Keep only the fields needed for downstream retrieval and result display.
final_df = retrieval_df[["doc_id", "parent_asin", "asin", "title", "rating", "text"]].copy()
final_df.head()

,doc_id,parent_asin,asin,title,rating,text
0,B00YQ6X8EO_0,B00YQ6X8EO,B00YQ6X8EO,Such a lovely scent but not overpowering.,5,such a lovely scent but not overpowering. this...
1,B081TJ8YS3_1,B081TJ8YS3,B081TJ8YS3,Works great but smells a little weird.,4,works great but smells a little weird. this pr...
2,B097R46CSY_2,B097R46CSY,B07PNNCSP9,Yes!,5,"yes! smells good, feels great!"
3,B09JS339BZ_3,B09JS339BZ,B09JS339BZ,Synthetic feeling,1,synthetic feeling felt synthetic
4,B08BZ63GMJ_4,B08BZ63GMJ,B08BZ63GMJ,A+,5,a+ love it


## Summary of EDA and preprocessing decisions

This notebook examined sampled review and metadata files from two candidate categories and selected **All_Beauty** for Milestone 1.

The main findings are as follows. First, the review data provides the most reliable retrieval content, with review text serving as the core searchable field and review titles adding short contextual information. Second, although metadata was explored during EDA, it was not consistently available after merging and was therefore not retained in the final cleaned retrieval dataset. Third, a compact document format built from identifiers, title, rating, and cleaned review text is appropriate for both BM25 and semantic retrieval. Finally, minimal preprocessing is preferred at this stage in order to preserve useful retrieval signal.

The resulting cleaned dataset will be used in later steps to build a **BM25 retriever**, a **semantic retriever** based on embeddings, and a simple search application. This keeps the milestone focused on retrieval preparation rather than generation.